<a href="https://colab.research.google.com/github/Aditi-k1912/Gen-AI/blob/main/Practical3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, set_seed)

set_seed(42)

# Small labelled news dataset (0 = Sports, 1 = Politics, 2 = Technology)
data = [
    ("India wins the cricket series against Australia", 0),
    ("Star striker scores a hat-trick in the league final", 0),
    ("Olympic champion breaks the 100 metre world record", 0),
    ("Mumbai Indians clinch the title in a last-over thriller", 0),
    ("Tennis player reaches the Wimbledon semi-final", 0),
    ("National hockey team qualifies for the World Cup", 0),
    ("Coach announces the squad for the upcoming football season", 0),
    ("Young wrestler wins gold at the Asian Games", 0),
    ("Badminton star defeats top seed in the championship", 0),
    ("Marathon runner sets a new national record", 0),
    ("Cricket board announces the schedule for the home series", 0),
    ("Defending champions beat rivals to enter the final", 0),
    ("Parliament passes the new budget after a long debate", 1),
    ("Prime Minister addresses the nation on economic reforms", 1),
    ("Opposition parties demand a discussion on the new bill", 1),
    ("Election Commission announces the polling dates", 1),
    ("Chief Minister inaugurates the new highway project", 1),
    ("Government introduces a policy to support farmers", 1),
    ("Senate votes on the proposed foreign policy changes", 1),
    ("Political leaders meet to form the coalition government", 1),
    ("Supreme Court hears the petition on the election law", 1),
    ("Minister announces new rules for state elections", 1),
    ("Lawmakers debate the amendment to the tax law", 1),
    ("President signs the education reform bill", 1),
    ("New smartphone launched with an advanced AI camera", 2),
    ("Tech company unveils its latest generative AI model", 2),
    ("Researchers develop a faster chip for data centres", 2),
    ("Cloud provider announces new cybersecurity tools", 2),
    ("Startup builds a robot that assists in surgery", 2),
    ("Software update brings new features to the operating system", 2),
    ("Scientists test a quantum computer with more qubits", 2),
    ("Electric car maker reveals a self-driving system", 2),
    ("New laptop offers longer battery life and faster graphics", 2),
    ("Social media platform adds an AI-powered search feature", 2),
    ("Engineers create a low-cost sensor for smart homes", 2),
    ("Company releases an open-source machine learning library", 2),
]
label_names = ["Sports", "Politics", "Technology"]

texts = [t for t, _ in data]
labels = [l for _, l in data]
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels)

# Load the pre-trained model and tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=3)

class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(texts, truncation=True,
                             padding=True, max_length=64)
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

train_ds = NewsDataset(X_train, y_train)
test_ds = NewsDataset(X_test, y_test)

args = TrainingArguments(output_dir="results",
                         num_train_epochs=6,
                         per_device_train_batch_size=8,
                         learning_rate=5e-5,
                         save_strategy="no",
                         report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=train_ds)
trainer.train()

# Evaluate on the test set
pred = trainer.predict(test_ds)
y_pred = np.argmax(pred.predictions, axis=1)
print("Test Accuracy:", round(accuracy_score(y_test, y_pred), 2))
print(classification_report(y_test, y_pred, target_names=label_names))

# Classify new articles
model.eval()
new_articles = [
    "The football team won the championship after a penalty shootout",
    "The government has proposed a new law to be debated in parliament",
    "The company launched a new AI chip for laptops",
]
for text in new_articles:
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    print(text, "->", label_names[int(logits.argmax())])


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(load

Step,Training Loss


Test Accuracy: 1.0
              precision    recall  f1-score   support

      Sports       1.00      1.00      1.00         3
    Politics       1.00      1.00      1.00         3
  Technology       1.00      1.00      1.00         3

    accuracy                           1.00         9
   macro avg       1.00      1.00      1.00         9
weighted avg       1.00      1.00      1.00         9

The football team won the championship after a penalty shootout -> Sports
The government has proposed a new law to be debated in parliament -> Politics
The company launched a new AI chip for laptops -> Technology
